# YOLO11 姿态估计应用案例

本案例基于 MindSpore 适配实现的 `ultralytics`，演示 YOLO11 姿态估计任务的训练、评估与推理完整流程。

In [33]:
import gc
import importlib
import os
import sys
import urllib.request
from pathlib import Path

import mindspore as ms

custom_ultralytics_parent = Path("/root/mindnlp/src/mindnlp")
if str(custom_ultralytics_parent) not in sys.path:
    sys.path.insert(0, str(custom_ultralytics_parent))

for module_name in list(sys.modules):
    if module_name == "ultralytics" or module_name.startswith("ultralytics."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import ultralytics
from ultralytics import YOLO

ms.set_context(mode=ms.PYNATIVE_MODE, device_target="Ascend")
package_dir = Path(ultralytics.__file__).resolve().parent
os.chdir(package_dir)
work_dir = Path.cwd() / "demo_inputs"
work_dir.mkdir(parents=True, exist_ok=True)

print("MindSpore version:", ms.__version__)
print("ultralytics package:", ultralytics.__file__)
print("package_dir:", package_dir)
print("cwd:", Path.cwd())

[WARNING] ME(43403:281472906070592,MainProcess):2026-04-24-17:05:14.369.000 [mindspore/context.py:1334] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


MindSpore version: 2.8.0
ultralytics package: /root/mindnlp/src/mindnlp/ultralytics/__init__.py
package_dir: /root/mindnlp/src/mindnlp/ultralytics
cwd: /root/mindnlp/src/mindnlp/ultralytics


## 1. 数据与测试图片准备

训练与评估默认使用 `cfg/datasets/coco8-pose.yaml`。推理阶段优先使用数据集中的样例图片；若当前环境下没有可用测试图片，则自动下载一张示例图片。

In [34]:
data_yaml = package_dir / "cfg/datasets/coco8-pose.yaml"
scratch_model_path = package_dir / "cfg/models/11/yolo11-pose.yaml"
finetune_model_path = package_dir / "yolo11n-pose.pt"

def resolve_source():
    candidates = [
        package_dir / "datasets/coco8-pose/images/val",
        package_dir / "datasets/coco8-pose/images/train",
    ]
    suffixes = {".jpg", ".jpeg", ".png", ".bmp"}
    for candidate in candidates:
        if candidate.is_file():
            return candidate
        if candidate.is_dir():
            files = sorted([p for p in candidate.rglob("*") if p.suffix.lower() in suffixes])
            if files:
                return files[0]

    image_path = work_dir / "pose_bus.jpg"
    if not image_path.exists():
        urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", image_path.as_posix())
        print("测试图片下载完成:", image_path)
    else:
        print("测试图片已存在:", image_path)
    return image_path

source_img = resolve_source()
print("数据配置:", data_yaml)
print("推理图片:", source_img)

数据配置: /root/mindnlp/src/mindnlp/ultralytics/cfg/datasets/coco8-pose.yaml
推理图片: /root/mindnlp/src/mindnlp/ultralytics/datasets/coco8-pose/images/val/000000000110.jpg


## 2. 模型训练

姿态估计任务支持使用预训练权重进行微调，也支持使用模型配置文件从头开始训练。

In [ ]:
# 微调
#model = YOLO(finetune_model_path.as_posix())
# 从头开始训练
model = YOLO(scratch_model_path.as_posix())

train_results = model.train(
    data=data_yaml.as_posix(),
    epochs=100,
    imgsz=640,
    batch=4,
    amp=False,
    val_interval=10,
    workers=8,
)

print("训练完成。")
print("best_fitness:", getattr(train_results, "best_fitness", None))
print("save_dir:", getattr(train_results, "save_dir", None))

[MindNLP YOLO] 检测到传入 YAML 架构文件: /root/mindnlp/src/mindnlp/ultralytics/cfg/models/11/yolo11-pose.yaml
[MindNLP YOLO] 模式: 从头开始随机初始化训练 (跳过权重转换)。
[MindNLP YOLO] 准备启动 pose 任务的训练...
[INFO] 训练任务启动，总轮数: 100 epochs
Epoch [0/99] Step [0/1] | Total Loss: 34.5890 | Loss Items: ['2.8692', '7.1322', '4.3780', '17.0229', '3.1869']
Epoch [1/99] Step [0/1] | Total Loss: 34.3706 | Loss Items: ['2.8571', '7.0832', '4.3145', '17.1740', '2.9418']
Epoch [2/99] Step [0/1] | Total Loss: 34.6868 | Loss Items: ['2.9101', '7.1914', '4.3728', '17.2227', '2.9899']
Epoch [3/99] Step [0/1] | Total Loss: 33.8326 | Loss Items: ['2.8574', '7.0667', '4.3347', '16.6608', '2.9130']
Epoch [4/99] Step [0/1] | Total Loss: 34.5702 | Loss Items: ['2.8795', '6.8740', '4.3825', '17.5432', '2.8911']
Epoch [5/99] Step [0/1] | Total Loss: 34.3739 | Loss Items: ['2.9292', '6.7925', '4.2711', '17.5326', '2.8485']
Epoch [6/99] Step [0/1] | Total Loss: 34.0572 | Loss Items: ['2.9036', '6.6867', '4.2602', '17.3303', '2.8763']
Epoch [7/9

Validating: 100%|██████████| 1/1 [00:06<00:00,  6.93s/it]
2026-04-24 19:06:13,819 - INFO - 推理测速: preprocess: 0.2ms | inference: 678.1ms | postprocess: 499.5ms


--------------------------------------------------
[评估报告] Epoch 9
  - mAP50(B)        : 0.00446
  - mAP50-95(B)     : 0.00180
  - mAP50(P)        : 0.00000
  - mAP50-95(P)     : 0.00000
[INFO] 当前模型综合评价指标 (Fitness): 0.00207
--------------------------------------------------

[INFO] 已更新最佳模型权重 (best.ckpt)，当前最高精度: 0.0021
Epoch [10/99] Step [0/1] | Total Loss: 32.2424 | Loss Items: ['2.7791', '6.0073', '4.2453', '16.5220', '2.6887']
Epoch [11/99] Step [0/1] | Total Loss: 32.2965 | Loss Items: ['2.8291', '5.9668', '4.2158', '16.7080', '2.5769']
Epoch [12/99] Step [0/1] | Total Loss: 31.7921 | Loss Items: ['2.7731', '5.8150', '4.1593', '16.4919', '2.5527']


## 3. 模型评估

训练完成后，在验证集上执行姿态估计评估。

In [36]:
val_results = model.val(
    data=data_yaml.as_posix(),
    imgsz=640,
    batch=4,
    workers=8,
    device="CPU"
)

print("评估完成。")
print(val_results)

[MindNLP YOLO] 准备启动 pose 任务的验证...


Validating: 100%|██████████| 1/1 [00:06<00:00,  6.42s/it]
2026-04-24 17:10:17,165 - INFO - 推理测速: preprocess: 0.2ms | inference: 783.3ms | postprocess: 92.8ms


评估完成。
{'speed': {'preprocess': 0.16415119171142578, 'inference': 783.311665058136, 'postprocess': 92.78059005737305}, 'metrics/mAP50(B)': 0.584898528902131, 'metrics/mAP50-95(B)': 0.4416917374380735, 'metrics/mAP50(P)': 0.3501689781021898, 'metrics/mAP50-95(P)': 0.15226905618226055, 'fitness': 0.6280714649587327}


## 4. 模型推理

推理结果会自动保存到输出目录，用户可直接查看可视化结果。

In [37]:
gc.collect()

predict_results = model(
    source=source_img.as_posix(),
    imgsz=640,
    conf=0.25,
    iou=0.45,
    save=True,
)

print("推理完成。")
if len(predict_results) > 0:
    print("推理结果保存目录:", getattr(predict_results[0], "save_dir", None))

2026-04-24 17:10:28,408 - INFO -  推理结果将保存至: /root/mindnlp/src/mindnlp/ultralytics/runs/detect/predict
2026-04-24 17:10:28,411 - INFO - 推理引擎启动，共探测到 1 份输入样本。


[MindNLP YOLO] 准备启动 pose 任务的推理...


2026-04-24 17:10:30,160 - INFO - 处理完成 [000000000110.jpg] | 前向推理: 1332.1ms | 后处理: 393.2ms


[Info] 渲染结果已保存至: runs/detect/predict/000000000110.jpg
推理完成。
推理结果保存目录: /root/mindnlp/src/mindnlp/ultralytics/runs/detect/predict
